# Experiment 3.3: Manifold Structure with Importance Decay

This notebook tests whether nonlinear encoders develop manifold structure when trained
on distributions with **constant sparsity** but **decaying importance** (I_i = 0.9^i).

Unlike the Zipf experiment (variable sparsity), here all features fire with equal probability,
but the loss weights features differently based on importance.

**Key questions:**
1. Does importance decay (without sparsity variation) promote manifold structure?
2. How does this compare to Zipf (sparsity decay) and uniform baselines?
3. Do low-importance features show different manifold properties than high-importance ones?

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torch import Generator

from occhio import ToyModel, MLPAutoencoder
from occhio.autoencoder import TiedLinear, TiedLinearRelu, TiedMLPEncoder
from occhio.distributions.sparse import SparseUniform
from occhio.analysis import (
    compute_feature_jacobians,
    angular_variance,
    jacobian_pca,
    direction_vs_context,
)

## 1. Configuration

- n=200 features, m=20 hidden (10:1 compression)
- Constant sparsity: p_active = 0.03 for all features
- Importance decay: I_i = 0.9^i

In [ ]:
# Configuration
N_FEATURES = 200
N_HIDDEN = 20  # 10:1 compression ratio
N_EPOCHS = 15000
BATCH_SIZE = 512
N_JACOBIAN_SAMPLES = 1000

# Constant sparsity (same as mean Zipf probability for comparison)
P_ACTIVE = 0.03

# Importance decay: I_i = 0.9^i
IMPORTANCE_DECAY = 0.996
IMPORTANCES = IMPORTANCE_DECAY ** torch.arange(N_FEATURES)

print(f"Configuration: {N_FEATURES} features -> {N_HIDDEN} hidden")
print(f"Compression ratio: {N_FEATURES / N_HIDDEN:.1f}:1")
print(f"Constant p_active: {P_ACTIVE}")
print(f"Importance range: [{IMPORTANCES[-1]:.6f}, {IMPORTANCES[0]:.6f}]")
print(f"Expected features per sample: {N_FEATURES * P_ACTIVE:.1f}")

In [ ]:
# Visualize importance decay
fig = go.Figure()
fig.add_trace(
    go.Bar(x=list(range(N_FEATURES)), y=IMPORTANCES.numpy(), name="Importance")
)
fig.update_layout(
    title=f"Feature Importances: I_i = {IMPORTANCE_DECAY}^i",
    xaxis_title="Feature Index",
    yaxis_title="Importance",
    yaxis_type="log",
    height=400,
)
fig.show()

## 2. Train Models

Training three architectures with importance decay:
1. **TiedLinear**: Linear baseline (angular variance should be ~0)
2. **TiedLinearRelu**: Piecewise linear (ReLU decoder)
3. **MLPAutoencoder**: Smooth nonlinearity (GELU encoder)

In [ ]:
def create_importance_distribution(seed=42):
    """Create SparseUniform with constant sparsity."""
    return SparseUniform(
        N_FEATURES,
        p_active=P_ACTIVE,
        generator=Generator().manual_seed(seed),
    )


def create_uniform_distribution(seed=42):
    """Create SparseUniform with uniform probabilities (no importance weighting)."""
    return SparseUniform(
        n_features=N_FEATURES,
        p_active=P_ACTIVE,
        generator=Generator().manual_seed(seed),
    )


# Verify distribution samples
dist = create_importance_distribution()
samples = dist.sample(1000)
print(f"Sample shape: {samples.shape}")
print(f"Feature 0 active rate: {(samples[:, 0] > 0).float().mean():.3f}")
print(f"Feature 100 active rate: {(samples[:, 100] > 0).float().mean():.3f}")
print(f"Feature 199 active rate: {(samples[:, 199] > 0).float().mean():.3f}")
print(f"Mean active features per sample: {(samples > 0).sum(dim=1).float().mean():.1f}")

In [ ]:
# Train TiedLinear (linear baseline) with importance decay
print("Training TiedLinear with importance decay...")
linear_model = ToyModel(
    distribution=create_importance_distribution(),
    ae=TiedLinear(n_features=N_FEATURES, n_hidden=N_HIDDEN),
    importances=IMPORTANCES,
)
linear_losses, _ = linear_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {linear_losses[-1]:.6f}")

In [ ]:
# Train TiedLinearRelu (piecewise linear) with importance decay
print("Training TiedLinearRelu with importance decay...")
relu_model = ToyModel(
    distribution=create_importance_distribution(),
    ae=TiedLinearRelu(n_features=N_FEATURES, n_hidden=N_HIDDEN),
    importances=IMPORTANCES,
)
relu_losses, _ = relu_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {relu_losses[-1]:.6f}")

In [ ]:
# Train MLPAutoencoder (smooth nonlinearity) with importance decay
print("Training MLPAutoencoder with importance decay...")
mlp_model = ToyModel(
    distribution=create_importance_distribution(),
    ae=TiedMLPEncoder(
        [N_FEATURES, N_HIDDEN * 4, N_HIDDEN],
    ),
    importances=IMPORTANCES,
)
mlp_losses, _ = mlp_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {mlp_losses[-1]:.6f}")

In [ ]:
# Train MLP WITHOUT importance decay for comparison
print("Training MLPAutoencoder WITHOUT importance decay (comparison)...")
mlp_uniform_model = ToyModel(
    distribution=create_uniform_distribution(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES,
        n_hidden=N_HIDDEN,
        encoder_hidden_dim=N_HIDDEN * 2,
        activation="gelu",
        decoder_activation="relu",
    ),
    # No importances - uniform weighting
)
mlp_uniform_losses, _ = mlp_uniform_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {mlp_uniform_losses[-1]:.6f}")

In [ ]:
# Plot training losses
fig = go.Figure()
fig.add_trace(go.Scatter(y=linear_losses, name="TiedLinear (Importance)", mode="lines"))
fig.add_trace(
    go.Scatter(y=relu_losses, name="TiedLinearRelu (Importance)", mode="lines")
)
fig.add_trace(go.Scatter(y=mlp_losses, name="MLP (Importance)", mode="lines"))
fig.add_trace(
    go.Scatter(
        y=mlp_uniform_losses, name="MLP (Uniform)", mode="lines", line=dict(dash="dash")
    )
)
fig.update_layout(
    title="Training Loss Comparison",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    yaxis_type="log",
    height=400,
)
fig.show()

## 3. Experiment A: Does Manifold Structure Emerge?

Compute angular variance for all features across models.
If MLP develops significantly higher AV than linear baseline, manifold structure is present.

In [ ]:
# Generate test samples for Jacobian analysis
test_dist = create_importance_distribution(seed=999)
test_samples = test_dist.sample(N_JACOBIAN_SAMPLES * 2)

# For uniform comparison
test_dist_uniform = create_uniform_distribution(seed=999)
test_samples_uniform = test_dist_uniform.sample(N_JACOBIAN_SAMPLES * 2)

print(f"Test samples shape: {test_samples.shape}")

In [ ]:
def compute_all_angular_variances(model, samples, n_samples_per_feature=500):
    """Compute angular variance for all features."""
    n_features = model.n_features
    avs = []

    for feat_idx in range(n_features):
        # Filter for samples where this feature is active
        active_mask = samples[:, feat_idx] > 0
        active_samples = samples[active_mask]

        if len(active_samples) < 10:
            avs.append(np.nan)
            continue

        active_samples = active_samples[:n_samples_per_feature]

        jacs = compute_feature_jacobians(model, feat_idx, active_samples)
        av = angular_variance(jacs)
        avs.append(av)

        if feat_idx % 50 == 0:
            print(f"  Feature {feat_idx}: AV = {av:.6f} (n={len(active_samples)})")

    return np.array(avs)


print("Computing angular variance for all features...")
print("\nTiedLinear (Importance):")
linear_avs = compute_all_angular_variances(linear_model, test_samples)

print("\nTiedLinearRelu (Importance):")
relu_avs = compute_all_angular_variances(relu_model, test_samples)

print("\nMLPAutoencoder (Importance):")
mlp_avs = compute_all_angular_variances(mlp_model, test_samples)

print("\nMLPAutoencoder (Uniform):")
mlp_uniform_avs = compute_all_angular_variances(mlp_uniform_model, test_samples_uniform)

In [ ]:
# Summary statistics (excluding NaN)
def summarize_avs(avs, name):
    valid = avs[~np.isnan(avs)]
    print(f"{name}:")
    print(f"  Mean AV: {valid.mean():.6f}")
    print(f"  Max AV:  {valid.max():.6f}")
    print(f"  Std AV:  {valid.std():.6f}")
    print(f"  Valid features: {len(valid)}/{len(avs)}")
    return valid


print("=" * 60)
print("ANGULAR VARIANCE SUMMARY")
print("=" * 60)
linear_valid = summarize_avs(linear_avs, "TiedLinear (Importance)")
print()
relu_valid = summarize_avs(relu_avs, "TiedLinearRelu (Importance)")
print()
mlp_valid = summarize_avs(mlp_avs, "MLPAutoencoder (Importance)")
print()
mlp_uniform_valid = summarize_avs(mlp_uniform_avs, "MLPAutoencoder (Uniform)")

In [ ]:
# Compute per-feature reconstruction loss (one-hot encoding)
def compute_one_hot_reconstruction_loss(model, n_features):
    """Compute reconstruction loss for each feature's one-hot encoding."""
    losses = []
    for i in range(n_features):
        one_hot = torch.zeros(
            1,
            n_features,
            device=model.ae.W.device
            if hasattr(model.ae, "W")
            else next(model.ae.parameters()).device,
        )
        one_hot[0, i] = 1.0
        with torch.no_grad():
            result = model.ae(one_hot)
            reconstructed = result[0] if isinstance(result, tuple) else result
            loss = ((one_hot - reconstructed) ** 2).sum().item()
        losses.append(loss)
    return np.array(losses)


# Compute one-hot reconstruction losses for MLP model
one_hot_losses = compute_one_hot_reconstruction_loss(mlp_model, N_FEATURES)
one_hot_losses_valid = one_hot_losses[valid_av_mask]

# Plot angular variance comparison
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Angular Variance by Feature (Importance Decay)",
        "Distribution Comparison",
        "AV vs Feature Importance",
        "Importance Decay vs Uniform MLP",
    ],
    specs=[[{}, {}], [{}, {}]],
)

features = list(range(N_FEATURES))

# Per-feature AV (Importance models)
fig.add_trace(
    go.Scatter(x=features, y=linear_avs, name="TiedLinear", mode="lines", opacity=0.7),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=features, y=relu_avs, name="TiedLinearRelu", mode="lines", opacity=0.7
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=features, y=mlp_avs, name="MLP (Importance)", mode="lines"),
    row=1,
    col=1,
)

# Box plot comparison
fig.add_trace(
    go.Box(y=linear_valid, name="TiedLinear", boxpoints="outliers"), row=1, col=2
)
fig.add_trace(
    go.Box(y=relu_valid, name="TiedLinearRelu", boxpoints="outliers"), row=1, col=2
)
fig.add_trace(
    go.Box(y=mlp_valid, name="MLP (Importance)", boxpoints="outliers"), row=1, col=2
)

# AV vs feature importance (scatter) - colored by one-hot reconstruction loss
fig.add_trace(
    go.Scatter(
        x=IMPORTANCES.numpy()[valid_av_mask],
        y=valid_avs,
        mode="markers",
        name="MLP (Importance)",
        marker=dict(
            size=5,
            opacity=0.6,
            color=one_hot_losses_valid,
            colorscale="rdbu",
            colorbar=dict(title="One-Hot Recon Loss"),
        ),
    ),
    row=2,
    col=1,
)

# Importance vs Uniform MLP comparison
fig.add_trace(
    go.Box(y=mlp_valid, name="MLP Importance", boxpoints="outliers"), row=2, col=2
)
fig.add_trace(
    go.Box(y=mlp_uniform_valid, name="MLP Uniform", boxpoints="outliers"), row=2, col=2
)

fig.update_xaxes(title_text="Feature Index", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=2)
fig.update_xaxes(title_text="Feature Importance", type="log", row=2, col=1)
fig.update_yaxes(title_text="Angular Variance", row=2, col=1)
fig.update_yaxes(title_text="Angular Variance", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Angular Variance Analysis: Importance Decay",
    showlegend=True,
)
fig.show()

In [ ]:
# Distribution of angular variance values (histogram/density)
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "AV Distribution: Importance vs Uniform",
        "AV Distribution (Log Scale)",
    ],
)

# Histogram comparison - use explicit colors for clarity
fig.add_trace(
    go.Histogram(
        x=mlp_valid,
        name="MLP (Importance)",
        opacity=0.6,
        nbinsx=30,
        marker_color="blue",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(
        x=mlp_uniform_valid,
        name="MLP (Uniform)",
        opacity=0.6,
        nbinsx=30,
        marker_color="orange",
    ),
    row=1,
    col=1,
)

# Log-scale histogram to see tail behavior - keep legend entries for second subplot
fig.add_trace(
    go.Histogram(
        x=np.log10(mlp_valid + 1e-8),
        name="MLP (Importance)",
        opacity=0.6,
        nbinsx=30,
        showlegend=False,
        marker_color="blue",
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Histogram(
        x=np.log10(mlp_uniform_valid + 1e-8),
        name="MLP (Uniform)",
        opacity=0.6,
        nbinsx=30,
        showlegend=False,
        marker_color="orange",
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Angular Variance", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_xaxes(title_text="log10(Angular Variance)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.update_layout(
    height=400,
    barmode="overlay",
    title_text="Angular Variance Distributions",
    legend=dict(x=0.85, y=0.95),
)
fig.show()

print(
    f"Importance decay AV: mean={np.nanmean(mlp_valid):.4f}, median={np.nanmedian(mlp_valid):.4f}, std={np.nanstd(mlp_valid):.4f}"
)
print(
    f"Uniform AV: mean={np.nanmean(mlp_uniform_valid):.4f}, median={np.nanmedian(mlp_uniform_valid):.4f}, std={np.nanstd(mlp_uniform_valid):.4f}"
)

## 4. Jacobian PCA Analysis

For features with high angular variance, analyze the dimensionality of the direction manifold.

In [ ]:
# Find features with highest angular variance in MLP
valid_mask = ~np.isnan(mlp_avs)
sorted_indices = np.argsort(mlp_avs[valid_mask])[::-1]
valid_features = np.where(valid_mask)[0]
top_av_features = valid_features[sorted_indices[:10]]

print("Top 10 features by angular variance (MLP Importance):")
for feat_idx in top_av_features:
    print(
        f"  Feature {feat_idx}: AV = {mlp_avs[feat_idx]:.6f}, importance = {IMPORTANCES[feat_idx]:.6f}"
    )

In [ ]:
# PCA analysis for top features
def analyze_feature_pca(model, feat_idx, samples, n_samples=500):
    """Compute PCA eigenvalues for a feature's Jacobian directions."""
    active_mask = samples[:, feat_idx] > 0
    active_samples = samples[active_mask][:n_samples]

    jacs = compute_feature_jacobians(model, feat_idx, active_samples)
    eigenvalues, eigenvectors = jacobian_pca(jacs)

    return eigenvalues.detach().numpy(), eigenvectors


# Compute PCA for top features
pca_results = {}
for feat_idx in top_av_features[:5]:
    eigenvalues, _ = analyze_feature_pca(mlp_model, feat_idx, test_samples)
    pca_results[feat_idx] = eigenvalues

    total_var = eigenvalues.sum()
    ratios = eigenvalues / total_var if total_var > 1e-10 else eigenvalues

    print(
        f"Feature {feat_idx} (AV={mlp_avs[feat_idx]:.4f}, importance={IMPORTANCES[feat_idx]:.4f}):"
    )
    print(f"  Top 5 eigenvalue ratios: {ratios[:5].round(4)}")
    print(f"  Cumulative variance (5 components): {ratios[:5].sum():.4f}")

In [ ]:
# Plot PCA eigenvalue spectra
fig = go.Figure()

for feat_idx, eigenvalues in pca_results.items():
    total_var = eigenvalues.sum()
    ratios = eigenvalues / total_var if total_var > 1e-10 else eigenvalues

    fig.add_trace(
        go.Bar(
            x=list(range(1, len(eigenvalues) + 1)),
            y=ratios,
            name=f"Feature {feat_idx} (AV={mlp_avs[feat_idx]:.3f})",
            opacity=0.7,
        )
    )

fig.update_layout(
    title="PCA Eigenvalue Spectrum for High-AV Features",
    xaxis_title="Principal Component",
    yaxis_title="Explained Variance Ratio",
    barmode="group",
    height=400,
)
fig.show()

## 5. Direction vs Context Analysis

For high-AV features, identify which co-active features cause the largest direction rotations.

In [ ]:
# Analyze direction vs context for the highest-AV feature
top_feat = top_av_features[0]
print(f"Analyzing feature {top_feat} (highest AV = {mlp_avs[top_feat]:.6f})")

# Get active samples
active_mask = test_samples[:, top_feat] > 0
active_samples = test_samples[active_mask][:500]

# Compute Jacobians
jacs = compute_feature_jacobians(mlp_model, top_feat, active_samples)

# Direction vs context
context_result = direction_vs_context(
    mlp_model,
    feature_idx=top_feat,
    inputs=active_samples,
    jacobians=jacs,
)

print(f"\nMost influential co-active features:")
for feat_idx, magnitude in context_result["most_influential"]:
    print(
        f"  Feature {feat_idx}: correlation magnitude = {magnitude:.4f}, importance = {IMPORTANCES[feat_idx]:.4f}"
    )

In [ ]:
# Visualize correlation magnitudes vs feature importance
corr_mags = context_result["correlation_magnitudes"].detach().numpy()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        f"Influence on Feature {top_feat}'s Direction",
        "Correlation Magnitude vs Importance",
    ],
)

# Bar chart
colors = ["red" if i == top_feat else "blue" for i in range(N_FEATURES)]
fig.add_trace(
    go.Bar(
        x=list(range(N_FEATURES)), y=corr_mags, marker_color=colors, showlegend=False
    ),
    row=1,
    col=1,
)

# Scatter: correlation vs importance
fig.add_trace(
    go.Scatter(
        x=IMPORTANCES.numpy(),
        y=corr_mags,
        mode="markers",
        marker=dict(size=5, color=corr_mags, colorscale="Viridis"),
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Feature Index", row=1, col=1)
fig.update_yaxes(title_text="Correlation Magnitude", row=1, col=1)
fig.update_xaxes(title_text="Feature Importance", type="log", row=1, col=2)
fig.update_yaxes(title_text="Correlation Magnitude", row=1, col=2)

fig.update_layout(
    height=400, title_text=f"Context Dependence Analysis for Feature {top_feat}"
)
fig.show()

## 6. 3D Visualization (Low-Dimensional Sanity Check)

Train models with n_hidden=3 for direct visualization of Jacobian directions on a sphere.

In [ ]:
# Smaller models for 3D visualization
N_FEATURES_3D = 50
N_HIDDEN_3D = 3
N_EPOCHS_3D = 2000

IMPORTANCES_3D = IMPORTANCE_DECAY ** torch.arange(N_FEATURES_3D)


def create_importance_distribution_3d(seed=42):
    return SparseUniform(
        N_FEATURES_3D,
        p_active=P_ACTIVE,
        generator=Generator().manual_seed(seed),
    )


print("Training 3D models...")

linear_3d = ToyModel(
    distribution=create_importance_distribution_3d(),
    ae=TiedLinear(n_features=N_FEATURES_3D, n_hidden=N_HIDDEN_3D),
    importances=IMPORTANCES_3D,
)
linear_3d.fit(n_epochs=N_EPOCHS_3D, batch_size=BATCH_SIZE)
print("Linear 3D trained.")

mlp_3d = ToyModel(
    distribution=create_importance_distribution_3d(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES_3D,
        n_hidden=N_HIDDEN_3D,
        encoder_hidden_dim=N_HIDDEN_3D * 2,
        activation="gelu",
        decoder_activation="relu",
    ),
    importances=IMPORTANCES_3D,
)
mlp_3d.fit(n_epochs=N_EPOCHS_3D, batch_size=BATCH_SIZE)
print("MLP 3D trained.")

In [ ]:
# Generate test samples for 3D models
test_dist_3d = create_importance_distribution_3d(seed=999)
test_samples_3d = test_dist_3d.sample(2000)

# Compute AV for all features in 3D models
print("Computing 3D angular variances...")
linear_3d_avs = []
mlp_3d_avs = []

for feat_idx in range(N_FEATURES_3D):
    active_mask = test_samples_3d[:, feat_idx] > 0
    active_samples = test_samples_3d[active_mask][:500]

    if len(active_samples) < 10:
        linear_3d_avs.append(np.nan)
        mlp_3d_avs.append(np.nan)
        continue

    linear_jacs = compute_feature_jacobians(linear_3d, feat_idx, active_samples)
    mlp_jacs = compute_feature_jacobians(mlp_3d, feat_idx, active_samples)

    linear_3d_avs.append(angular_variance(linear_jacs))
    mlp_3d_avs.append(angular_variance(mlp_jacs))

linear_3d_avs = np.array(linear_3d_avs)
mlp_3d_avs = np.array(mlp_3d_avs)

print(f"Linear 3D mean AV: {np.nanmean(linear_3d_avs):.6f}")
print(f"MLP 3D mean AV: {np.nanmean(mlp_3d_avs):.6f}")

In [ ]:
# Find highest-AV feature in 3D MLP
top_feat_3d = np.nanargmax(mlp_3d_avs)
print(f"Visualizing feature {top_feat_3d} (AV = {mlp_3d_avs[top_feat_3d]:.6f})")

# Get samples and compute Jacobians
active_mask = test_samples_3d[:, top_feat_3d] > 0
active_samples_3d = test_samples_3d[active_mask][:500]

linear_jacs_3d = compute_feature_jacobians(linear_3d, top_feat_3d, active_samples_3d)
mlp_jacs_3d = compute_feature_jacobians(mlp_3d, top_feat_3d, active_samples_3d)

# Normalize to unit sphere
linear_normed = linear_jacs_3d / linear_jacs_3d.norm(dim=1, keepdim=True).clamp(
    min=1e-8
)
mlp_normed = mlp_jacs_3d / mlp_jacs_3d.norm(dim=1, keepdim=True).clamp(min=1e-8)

# 3D scatter plot
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=["TiedLinear (should cluster)", "MLPAutoencoder (may spread)"],
)

# Linear model
fig.add_trace(
    go.Scatter3d(
        x=linear_normed[:, 0].detach().cpu().numpy(),
        y=linear_normed[:, 1].detach().cpu().numpy(),
        z=linear_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(size=3, color="blue", opacity=0.6),
        name="Linear",
    ),
    row=1,
    col=1,
)

# MLP model - color by most common co-active feature
ctx = direction_vs_context(mlp_3d, top_feat_3d, active_samples_3d, mlp_jacs_3d)
influential_feat = ctx["most_influential"][0][0] if ctx["most_influential"] else 1
colors = active_samples_3d[:, influential_feat].numpy()

fig.add_trace(
    go.Scatter3d(
        x=mlp_normed[:, 0].detach().cpu().numpy(),
        y=mlp_normed[:, 1].detach().cpu().numpy(),
        z=mlp_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(
            size=3,
            color=colors,
            colorscale="Viridis",
            colorbar=dict(title=f"Feature {influential_feat}", x=1.0),
            opacity=0.6,
        ),
        name="MLP",
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title=f"Jacobian Directions for Feature {top_feat_3d} on Unit Sphere (Importance Decay)",
    height=500,
    showlegend=False,
)
fig.show()

## 7. Experiment C: Reconstruction Quality Comparison

Compare final reconstruction loss. If MLP achieves significantly lower loss,
the nonlinear encoding is doing useful work.

In [ ]:
# Compute reconstruction loss on held-out data
test_dist_eval = create_importance_distribution(seed=12345)
eval_samples = test_dist_eval.sample(5000)


def compute_reconstruction_loss(model, samples):
    with torch.no_grad():
        result = model.ae(samples)
        reconstructed = result[0] if isinstance(result, tuple) else result
        mse = ((samples - reconstructed) ** 2).mean().item()
    return mse


linear_loss = compute_reconstruction_loss(linear_model, eval_samples)
relu_loss = compute_reconstruction_loss(relu_model, eval_samples)
mlp_loss = compute_reconstruction_loss(mlp_model, eval_samples)

# Uniform comparison
eval_samples_uniform = create_uniform_distribution(seed=12345).sample(5000)
mlp_uniform_loss = compute_reconstruction_loss(mlp_uniform_model, eval_samples_uniform)

print("=" * 60)
print("RECONSTRUCTION LOSS COMPARISON (held-out data)")
print("=" * 60)
print(f"TiedLinear (Importance):      {linear_loss:.6f}")
print(f"TiedLinearRelu (Importance):  {relu_loss:.6f}")
print(f"MLPAutoencoder (Importance):  {mlp_loss:.6f}")
print(f"MLPAutoencoder (Uniform):     {mlp_uniform_loss:.6f}")
print()
print(
    f"MLP vs Linear improvement (Importance): {(linear_loss - mlp_loss) / linear_loss * 100:.2f}%"
)
print()
print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Compression ratio: {N_FEATURES}:{N_HIDDEN} = {N_FEATURES / N_HIDDEN:.1f}:1")
print(f"Sparsity (p_active): {P_ACTIVE}")
print(f"Expected active features per sample: {N_FEATURES * P_ACTIVE:.1f}")

## 7b. Per-Feature Reconstruction Analysis

Key question: Does the MLP's 5.45% global improvement come specifically from
features with high angular variance, or is it spread uniformly?

In [ ]:
def compute_per_feature_reconstruction(model, samples):
    """Compute MSE for each feature across samples."""
    with torch.no_grad():
        result = model.ae(samples)
        reconstructed = result[0] if isinstance(result, tuple) else result
        # Per-feature MSE: average squared error for each feature dimension
        per_feature_mse = ((samples - reconstructed) ** 2).mean(dim=0)
    return per_feature_mse.numpy()


# Compute per-feature reconstruction for all models
linear_per_feat = compute_per_feature_reconstruction(linear_model, eval_samples)
mlp_per_feat = compute_per_feature_reconstruction(mlp_model, eval_samples)
mlp_uniform_per_feat = compute_per_feature_reconstruction(
    mlp_uniform_model, eval_samples_uniform
)

# Compute improvement per feature
improvement_per_feat = (
    (linear_per_feat - mlp_per_feat) / np.maximum(linear_per_feat, 1e-10) * 100
)

print("Per-feature reconstruction analysis:")
print(f"  Features where MLP improves: {(improvement_per_feat > 0).sum()}/{N_FEATURES}")
print(f"  Features where Linear wins:  {(improvement_per_feat < 0).sum()}/{N_FEATURES}")
print(f"  Mean improvement: {improvement_per_feat.mean():.2f}%")
print(f"  Max improvement:  {improvement_per_feat.max():.2f}%")
print(f"  Max degradation:  {improvement_per_feat.min():.2f}%")

In [ ]:
# Correlation between angular variance and reconstruction improvement
# This is the key test: if high-AV features show better reconstruction improvement,
# manifold structure is doing useful work

from scipy import stats

# Filter out NaN angular variances
valid_av_mask = ~np.isnan(mlp_avs)
valid_avs = mlp_avs[valid_av_mask]
valid_improvements = improvement_per_feat[valid_av_mask]

correlation, p_value = stats.pearsonr(valid_avs, valid_improvements)
spearman_corr, spearman_p = stats.spearmanr(valid_avs, valid_improvements)

print("=" * 60)
print("CORRELATION: Angular Variance vs Reconstruction Improvement")
print("=" * 60)
print(f"Pearson correlation:  r = {correlation:.4f}, p = {p_value:.4e}")
print(
    f"Spearman correlation: ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ = {spearman_corr:.4f}, p = {spearman_p:.4e}"
)
print()

if correlation > 0.2 and p_value < 0.05:
    print("=> POSITIVE correlation: High-AV features show better reconstruction!")
    print("   Manifold structure is doing useful work.")
elif correlation < -0.2 and p_value < 0.05:
    print("=> NEGATIVE correlation: High-AV features reconstruct worse.")
    print("   Manifold structure may be parasitic.")
else:
    print("=> No significant correlation detected.")
    print("   MLP improvement is spread across features regardless of AV.")

In [ ]:
# Visualize per-feature reconstruction and AV correlation
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Per-Feature MSE: Linear vs MLP",
        "MLP Improvement by Feature",
        "Angular Variance vs Reconstruction Improvement",
        "Improvement vs Feature Importance",
    ],
)

# Per-feature MSE comparison
fig.add_trace(
    go.Scatter(
        x=list(range(N_FEATURES)),
        y=linear_per_feat,
        name="Linear",
        mode="lines",
        opacity=0.7,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=list(range(N_FEATURES)), y=mlp_per_feat, name="MLP", mode="lines"),
    row=1,
    col=1,
)

# Improvement per feature
colors = ["green" if x > 0 else "red" for x in improvement_per_feat]
fig.add_trace(
    go.Bar(
        x=list(range(N_FEATURES)),
        y=improvement_per_feat,
        marker_color=colors,
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

# AV vs Improvement scatter (the key plot)
fig.add_trace(
    go.Scatter(
        x=valid_avs,
        y=valid_improvements,
        mode="markers",
        marker=dict(
            size=6,
            color=IMPORTANCES.numpy()[valid_av_mask],
            colorscale="Viridis",
            colorbar=dict(title="Importance", x=1.15),
        ),
        showlegend=False,
        text=[f"Feature {i}" for i in np.where(valid_av_mask)[0]],
        hovertemplate="Feature %{text}<br>AV: %{x:.4f}<br>Improvement: %{y:.2f}%<extra></extra>",
    ),
    row=2,
    col=1,
)
# Add trendline
z = np.polyfit(valid_avs, valid_improvements, 1)
p = np.poly1d(z)
x_trend = np.linspace(valid_avs.min(), valid_avs.max(), 100)
fig.add_trace(
    go.Scatter(
        x=x_trend,
        y=p(x_trend),
        mode="lines",
        line=dict(dash="dash", color="red"),
        name=f"Trend (r={correlation:.2f})",
        showlegend=True,
    ),
    row=2,
    col=1,
)

# Improvement vs Importance
fig.add_trace(
    go.Scatter(
        x=IMPORTANCES.numpy(),
        y=improvement_per_feat,
        mode="markers",
        marker=dict(size=6, color=mlp_avs, colorscale="Plasma"),
        showlegend=False,
    ),
    row=2,
    col=2,
)

fig.update_xaxes(title_text="Feature Index", row=1, col=1)
fig.update_yaxes(title_text="MSE", row=1, col=1)
fig.update_xaxes(title_text="Feature Index", row=1, col=2)
fig.update_yaxes(title_text="Improvement (%)", row=1, col=2)
fig.update_xaxes(title_text="Angular Variance", row=2, col=1)
fig.update_yaxes(title_text="Reconstruction Improvement (%)", row=2, col=1)
fig.update_xaxes(title_text="Feature Importance", type="log", row=2, col=2)
fig.update_yaxes(title_text="Reconstruction Improvement (%)", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Per-Feature Reconstruction Analysis: Does AV Predict Improvement?",
)
fig.show()

In [ ]:
# Breakdown: Which features contribute most to the 5.45% improvement?
# Group by angular variance quartiles

av_quartiles = np.percentile(valid_avs, [25, 50, 75])
quartile_labels = [
    "Low AV (Q1)",
    "Medium-Low AV (Q2)",
    "Medium-High AV (Q3)",
    "High AV (Q4)",
]


def get_quartile(av):
    if av < av_quartiles[0]:
        return 0
    elif av < av_quartiles[1]:
        return 1
    elif av < av_quartiles[2]:
        return 2
    else:
        return 3


quartile_assignments = np.array([get_quartile(av) for av in valid_avs])

print("=" * 60)
print("RECONSTRUCTION IMPROVEMENT BY ANGULAR VARIANCE QUARTILE")
print("=" * 60)
for q in range(4):
    mask = quartile_assignments == q
    q_improvements = valid_improvements[mask]
    q_avs = valid_avs[mask]
    print(f"\n{quartile_labels[q]}:")
    print(f"  N features: {mask.sum()}")
    print(f"  AV range: [{q_avs.min():.4f}, {q_avs.max():.4f}]")
    print(f"  Mean improvement: {q_improvements.mean():.2f}%")
    print(f"  Features improving: {(q_improvements > 0).sum()}/{mask.sum()}")

## 8. Interpreting the Two-Cluster Structure: Cooperative vs Parasitic

The strong negative correlation (r = -0.81) between AV and reconstruction improvement
admits two interpretations:

**Parasitic hypothesis**: High-AV features are dysfunction. The bending "corrupts"
their representation and they reconstruct poorly as a result.

**Cooperative hypothesis**: The MLP makes a *global* trade-off. Some features sacrifice
their own reconstruction quality (accepting higher AV) so that other features can be
represented much better. The bending isn't failure ﷿﷿﷿﷿﷿﷿﷿﷿﷿ it's accommodation.

The global numbers support the cooperative reading: the MLP is ~35% better overall.
If bending were purely parasitic, the losses from high-AV features would drag down
the total. Instead, the gains from low-AV features more than compensate.

**Critical test**: Do the high-AV features have high interference with the low-AV
features *in the linear model*? If the features that bend are exactly the ones
that would cause the most interference with the well-reconstructed features,
that's cooperative. If there's no relationship, it's more likely optimization noise.

In [ ]:
# Test: Do high-AV features interfere most with low-AV features in the linear model?

# Compute interference matrix from linear model's embedding weights
# Interference_ij = |W_i ﷿﷿﷿﷿﷿﷿ W_j| / (||W_i|| ||W_j||) for i != j
linear_W = linear_model.ae.W.detach()  # Shape: (n_hidden, n_features)
W_normed = linear_W / linear_W.norm(dim=0, keepdim=True).clamp(
    min=1e-8
)  # Normalize columns
interference_matrix = (W_normed.T @ W_normed).abs()  # Cosine similarity magnitude
interference_matrix.fill_diagonal_(0)  # Zero out self-interference

# For each feature, compute total interference with well-reconstructed features (low AV)
# Hypothesis: high-AV features should have high interference with low-AV features if cooperative

# Identify low-AV and high-AV feature sets
low_av_mask = mlp_avs < np.percentile(mlp_avs[~np.isnan(mlp_avs)], 25)
high_av_mask = mlp_avs > np.percentile(mlp_avs[~np.isnan(mlp_avs)], 75)

low_av_indices = np.where(low_av_mask)[0]
high_av_indices = np.where(high_av_mask)[0]

print(f"Low AV features (Q1): {len(low_av_indices)}")
print(f"High AV features (Q4): {len(high_av_indices)}")

# For each high-AV feature, compute mean interference with low-AV features
high_av_interference_with_low = []
for idx in high_av_indices:
    interf = interference_matrix[idx, low_av_indices].mean().item()
    high_av_interference_with_low.append(interf)

# For each low-AV feature, compute mean interference with other low-AV features
low_av_interference_with_low = []
for idx in low_av_indices:
    others = [i for i in low_av_indices if i != idx]
    interf = interference_matrix[idx, others].mean().item()
    low_av_interference_with_low.append(interf)

print(
    f"\nMean interference of HIGH-AV features with LOW-AV features: {np.mean(high_av_interference_with_low):.4f}"
)
print(
    f"Mean interference of LOW-AV features with other LOW-AV features: {np.mean(low_av_interference_with_low):.4f}"
)

# Statistical test
from scipy import stats as sp_stats

t_stat, t_pval = sp_stats.ttest_ind(
    high_av_interference_with_low, low_av_interference_with_low
)
print(f"\nt-test: t = {t_stat:.3f}, p = {t_pval:.4e}")

if (
    np.mean(high_av_interference_with_low) > np.mean(low_av_interference_with_low)
    and t_pval < 0.05
):
    print(
        "\n=> COOPERATIVE: High-AV features would have interfered most with low-AV features!"
    )
    print(
        "   The MLP is making these features 'bend out of the way' for the global good."
    )
else:
    print("\n=> No significant difference in interference patterns.")

In [ ]:
# Deeper analysis: correlation between a feature's AV and its interference with low-AV features
# If cooperative: features with higher interference -> higher AV (they bend to accommodate)

# For all features, compute mean interference with the low-AV feature cluster
interference_with_low_av = []
for idx in range(N_FEATURES):
    interf = interference_matrix[idx, low_av_indices].mean().item()
    interference_with_low_av.append(interf)
interference_with_low_av = np.array(interference_with_low_av)

# Correlation: AV vs interference with well-reconstructed features
valid_interference = interference_with_low_av[valid_av_mask]
corr_av_interference, p_av_interference = sp_stats.pearsonr(
    valid_avs, valid_interference
)
print(f"Correlation (AV vs interference with low-AV features):")
print(f"  Pearson r = {corr_av_interference:.4f}, p = {p_av_interference:.4e}")

# Also test: does interference predict which features the MLP "sacrifices"?
# I.e., features that would cause interference become the high-AV, worse-reconstruction features
corr_interf_improvement, p_interf_improvement = sp_stats.pearsonr(
    valid_interference, valid_improvements
)
print(f"\nCorrelation (interference with low-AV vs reconstruction improvement):")
print(f"  Pearson r = {corr_interf_improvement:.4f}, p = {p_interf_improvement:.4e}")

if corr_interf_improvement < -0.2 and p_interf_improvement < 0.05:
    print(
        "\n=> COOPERATIVE confirmed: Features that interfere with the 'winners' get sacrificed."
    )
    print(
        "   The MLP learns which features to make flexible to reduce global interference."
    )
elif corr_av_interference > 0.2 and p_av_interference < 0.05:
    print(
        "\n=> Partial support for cooperative: High-interference features develop high AV."
    )
else:
    print("\n=> Interference doesn't predict which features become sacrificial.")

In [ ]:
# Visualize the interference analysis
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Interference with Low-AV Features",
        "AV vs Interference",
        "Reconstruction Improvement vs Interference",
    ],
)

# Box plot: High-AV vs Low-AV interference with low-AV cluster
fig.add_trace(
    go.Box(
        y=high_av_interference_with_low,
        name="High-AV features",
        marker_color="red",
        boxpoints="all",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Box(
        y=low_av_interference_with_low,
        name="Low-AV features",
        marker_color="green",
        boxpoints="all",
    ),
    row=1,
    col=1,
)

# Scatter: AV vs interference
fig.add_trace(
    go.Scatter(
        x=valid_interference,
        y=valid_avs,
        mode="markers",
        marker=dict(
            size=5,
            color=valid_improvements,
            colorscale="RdYlGn",
            colorbar=dict(title="Improvement (%)", x=0.65),
        ),
        showlegend=False,
        hovertemplate="Interference: %{x:.4f}<br>AV: %{y:.4f}<extra></extra>",
    ),
    row=1,
    col=2,
)

# Scatter: improvement vs interference
fig.add_trace(
    go.Scatter(
        x=valid_interference,
        y=valid_improvements,
        mode="markers",
        marker=dict(
            size=5,
            color=valid_avs,
            colorscale="Plasma",
            colorbar=dict(title="Angular Var", x=1.02),
        ),
        showlegend=False,
        hovertemplate="Interference: %{x:.4f}<br>Improvement: %{y:.2f}%<extra></extra>",
    ),
    row=1,
    col=3,
)

fig.update_xaxes(title_text="Feature Group", row=1, col=1)
fig.update_yaxes(title_text="Mean Interference with Low-AV", row=1, col=1)
fig.update_xaxes(title_text="Interference with Low-AV Features", row=1, col=2)
fig.update_yaxes(title_text="Angular Variance", row=1, col=2)
fig.update_xaxes(title_text="Interference with Low-AV Features", row=1, col=3)
fig.update_yaxes(title_text="Reconstruction Improvement (%)", row=1, col=3)

fig.update_layout(
    height=400,
    title_text="Cooperative Trade-off Analysis: Does Linear Interference Predict MLP Sacrifice?",
    showlegend=True,
)
fig.show()

### Interpretation: Dimensionality Budget and Dynamic Reallocation

The linear model assigns fixed directions to features. Competition for limited dimensions
creates interference. Features that point in similar directions hurt each other's reconstruction.

The MLP can do something the linear model can't: **dynamically reassign encoding directions
depending on context**. A "sacrificial" feature doesn't need a fixed direction ﷿﷿﷿﷿﷿﷿﷿﷿﷿ it just
needs to stay out of the way of whatever else is active.

The circular/spread patterns we see for high-AV features may be exactly this: they're not
encoding useful information about themselves, they're *clearing space* by rotating to
wherever creates least interference with the currently active features.

If this interpretation is correct, we'd expect:
1. High-AV features to have high interference with low-AV features in the linear embedding
2. The MLP to achieve its gains specifically by improving low-AV features (which it does: ~70% improvement for Q1/Q2)
3. High-AV features to accept worse reconstruction as the "cost" of providing this flexibility

## 7c. Higher Compression Ratios

Test whether MLP advantage grows with increased compression. Currently at 10:1;
try 20:1 and 40:1 to see if manifold structure becomes more beneficial.

In [ ]:
# Test higher compression ratios
# This addresses the question: "You might be near the bottom of a curve where
# the MLP advantage grows rapidly with compression"

COMPRESSION_RATIOS = [10, 20, 40]  # N_FEATURES / N_HIDDEN
N_EPOCHS_COMPRESSION = 10000  # Slightly fewer epochs for faster iteration

compression_results = {}

for ratio in COMPRESSION_RATIOS:
    n_hidden = N_FEATURES // ratio
    print(f"\n{'=' * 60}")
    print(f"Testing {ratio}:1 compression ({N_FEATURES} -> {n_hidden})")
    print(f"{'=' * 60}")

    # Train linear model
    linear = ToyModel(
        distribution=create_uniform_distribution(),
        ae=TiedLinear(n_features=N_FEATURES, n_hidden=n_hidden),
    )
    linear.fit(n_epochs=N_EPOCHS_COMPRESSION, batch_size=BATCH_SIZE)

    # Train MLP model
    mlp = ToyModel(
        distribution=create_uniform_distribution(),
        ae=MLPAutoencoder(
            n_features=N_FEATURES,
            n_hidden=n_hidden,
            encoder_hidden_dim=max(n_hidden * 2, 20),
            activation="gelu",
            decoder_activation="relu",
        ),
    )
    mlp.fit(n_epochs=N_EPOCHS_COMPRESSION, batch_size=BATCH_SIZE)

    # Evaluate on held-out data
    eval_dist = create_uniform_distribution(seed=54321)
    eval_data = eval_dist.sample(5000)

    linear_mse = compute_reconstruction_loss(linear, eval_data)
    mlp_mse = compute_reconstruction_loss(mlp, eval_data)
    improvement = (linear_mse - mlp_mse) / linear_mse * 100

    # Compute mean AV for MLP
    test_data = eval_dist.sample(2000)
    avs = []
    for feat_idx in range(0, N_FEATURES, 10):  # Sample every 10th feature for speed
        active_mask = test_data[:, feat_idx] > 0
        active_samples = test_data[active_mask][:200]
        if len(active_samples) >= 10:
            jacs = compute_feature_jacobians(mlp, feat_idx, active_samples)
            avs.append(angular_variance(jacs))
    mean_av = np.mean(avs)

    compression_results[ratio] = {
        "linear_mse": linear_mse,
        "mlp_mse": mlp_mse,
        "improvement": improvement,
        "mean_av": mean_av,
    }

    print(f"  Linear MSE: {linear_mse:.6f}")
    print(f"  MLP MSE:    {mlp_mse:.6f}")
    print(f"  MLP improvement: {improvement:.2f}%")
    print(f"  Mean AV (sampled): {mean_av:.6f}")

In [ ]:
# Plot compression ratio results
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "MLP Improvement vs Compression Ratio",
        "Mean AV vs Compression Ratio",
    ],
)

ratios = list(compression_results.keys())
improvements = [compression_results[r]["improvement"] for r in ratios]
mean_avs = [compression_results[r]["mean_av"] for r in ratios]

fig.add_trace(
    go.Bar(x=[f"{r}:1" for r in ratios], y=improvements, marker_color="green"),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(x=[f"{r}:1" for r in ratios], y=mean_avs, marker_color="purple"),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Compression Ratio", row=1, col=1)
fig.update_yaxes(title_text="MLP Improvement (%)", row=1, col=1)
fig.update_xaxes(title_text="Compression Ratio", row=1, col=2)
fig.update_yaxes(title_text="Mean Angular Variance", row=1, col=2)

fig.update_layout(
    height=400,
    title_text="Effect of Compression Ratio on Manifold Structure",
    showlegend=False,
)
fig.show()

print("\nCompression ratio summary:")
print("=" * 60)
for r in ratios:
    res = compression_results[r]
    print(
        f"{r}:1 -> Improvement: {res['improvement']:.2f}%, Mean AV: {res['mean_av']:.4f}"
    )

In [ ]:
print("=" * 70)
print("EXPERIMENT 3.3 SUMMARY: Manifold Structure with Importance Decay")
print("=" * 70)
print()
print("CONFIGURATION:")
print(f"  Compression ratio: {N_FEATURES}:{N_HIDDEN} = {N_FEATURES / N_HIDDEN:.0f}:1")
print(f"  Sparsity: p_active = {P_ACTIVE}")
print(f"  Expected active features/sample: {N_FEATURES * P_ACTIVE:.1f}")
print()
print("1. ANGULAR VARIANCE (manifold structure detection):")
print(f"   - Linear baseline: {np.nanmean(linear_avs):.6f} (expected ~0)")
print(f"   - MLP (Importance): {np.nanmean(mlp_avs):.6f}")
print(f"   - MLP (Uniform):    {np.nanmean(mlp_uniform_avs):.6f}")
print()

av_increase = np.nanmean(mlp_avs) / max(np.nanmean(linear_avs), 1e-10)
if av_increase > 10:
    print("   => SIGNIFICANT manifold structure detected in MLP!")
elif av_increase > 2:
    print("   => Moderate manifold structure detected.")
else:
    print("   => MLP converged to near-linear encoding.")

print()
print("2. RECONSTRUCTION QUALITY:")
print(f"   - Linear: {linear_loss:.6f}")
print(f"   - MLP:    {mlp_loss:.6f}")
improvement = (linear_loss - mlp_loss) / linear_loss * 100
print(f"   - Improvement: {improvement:.2f}%")

if improvement > 10:
    print("   => MLP's nonlinearity provides substantial benefit!")
elif improvement > 2:
    print("   => MLP provides modest improvement.")
else:
    print("   => Linearity is near-optimal for this task.")

print()
print("3. PER-FEATURE RECONSTRUCTION BY AV QUARTILE:")
for q in range(4):
    mask = quartile_assignments == q
    q_improvements = valid_improvements[mask]
    print(f"   - {quartile_labels[q]}: {q_improvements.mean():.1f}% improvement")

print()
print("4. COOPERATIVE vs PARASITIC INTERPRETATION:")
print(f"   - Pearson r (AV vs improvement): {correlation:.4f}")
print(f"   => Strong negative correlation: high-AV features reconstruct worse")
print()
print("   Two possible readings:")
print("   a) PARASITIC: Bending corrupts representation (dysfunction)")
print("   b) COOPERATIVE: Features sacrifice themselves for global good")
print()
print("   Evidence for cooperative interpretation:")
print(f"   - Global MLP improvement: {improvement:.1f}% (gains exceed losses)")
print(f"   - Q1/Q2 (low AV): ~70% improvement (the 'winners')")
print(f"   - Q4 (high AV): ~-14% (the 'sacrifices')")
print(f"   - High-AV vs Low-AV interference with winners:")
print(f"       High-AV mean: {np.mean(high_av_interference_with_low):.4f}")
print(f"       Low-AV mean:  {np.mean(low_av_interference_with_low):.4f}")

if np.mean(high_av_interference_with_low) > np.mean(low_av_interference_with_low):
    print("   => High-AV features had higher interference with winners!")
    print("      Supports cooperative: they 'bend out of the way'")
else:
    print("   => No interference difference; mechanism unclear")

print()
print("5. COMPRESSION RATIO SCALING:")
if compression_results:
    for r in sorted(compression_results.keys()):
        res = compression_results[r]
        print(
            f"   - {r}:1 compression: {res['improvement']:.2f}% improvement, AV={res['mean_av']:.4f}"
        )

print()
print("=" * 70)
print("KEY INSIGHT")
print("=" * 70)
print("""
The two-cluster structure (low-AV/high-improvement vs high-AV/low-improvement)
is consistent with the MLP making a GLOBAL TRADE-OFF:

- Some features get dedicated, stable encoding directions (low AV)
  and achieve excellent reconstruction (~70% better than linear)
  
- Other features become "flexible" (high AV), rotating their encoding
  directions based on context to reduce interference with the winners.
  They accept worse personal reconstruction for the global good.

This is NOT failure ﷿﷿﷿﷿﷿﷿﷿﷿﷿ it's a dimensionality allocation strategy that
the linear model cannot employ. The MLP achieves 35% overall improvement
precisely BECAUSE it can make this trade-off.
""")